# AeroPulse PM2.5 — Driver Acquisition: satellite chemistry, terrain, calendar

Acquire the three driver layers the Phase-2 feature set is missing: CAMS atmospheric
composition (AOD, NO2, CO, SO2, dust, PM10), station terrain, and the Indian
festival / crop-burning calendar.

## Objective

Phase 5 diagnosed the model's central defect as **extreme-event underprediction**
(bias −103 µg/m³ on the `extreme` regime). The roadmap attributes this partly to the
loss function and partly to missing physical information. This notebook acquires the
missing information.

Three layers, three very different reliability characteristics — which is why they are
acquired separately and labelled separately:

| Layer | Source | What it is |
|---|---|---|
| Atmospheric composition | Open-Meteo CAMS reanalysis | **model output**, not observation |
| Terrain | Open-Meteo elevation (Copernicus DEM) | measured, static |
| Calendar | curated table in this notebook | deterministic, human-authored |

### The CAMS caveat is load-bearing

`cams_*` columns are **CAMS-derived model output, not ground truth**. CAMS is itself a
chemical transport model that assimilates satellite radiances; its PM2.5 field is a
*prediction*, not a measurement. Two rules follow, and both are enforced below:

1. `cams_*` may be used as a **predictor** only. It is never a target.
2. No metric computed against CAMS may be described as station accuracy.

`cams_pm25` is acquired for diagnostics — it is genuinely useful to know how well a
physics model does on the same stations — but it is **written into a separate audit
column and excluded from the feature contract**. Feeding a model another model's PM2.5
estimate would produce a shortcut that scores well and transfers nowhere.

### What this notebook does not do

Population density, road density, land use and building density are in the roadmap
(§15) and are **not** acquired here: they need WorldPop / GHSL / OSM bulk downloads
rather than a point API. They are recorded as an open gap in the manifest rather than
silently skipped.

### Outputs

```text
data/pm25/raw/satellite/cams_{lat}_{lon}.parquet   per-coordinate cache
data/pm25/processed/base/satellite_hourly.parquet  canonical hourly composition
data/pm25/processed/base/static_geography.parquet  one row per station
data/pm25/processed/base/calendar_events.parquet   one row per local date
artifacts/pm25/phase7/driver_acquisition_manifest.json
```

In [1]:
# ============================================================================
# CONFIGURATION - paths and acquisition scope
# ============================================================================
# Same layout as Phase 1 so every notebook in this folder resolves the same
# tree. The window is DERIVED from the Phase-1 dataset rather than configured:
# a driver layer that does not span the target layer produces silent NaNs in
# Phase 2, and hardcoding dates is how those two drift apart.
# ============================================================================
from pathlib import Path
import os, io, json, math, time, warnings, hashlib
import numpy as np
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

_dotenv_path = find_dotenv(usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path, override=True)
    print("Loaded env from:", _dotenv_path)
else:
    # No key is required by any source in this notebook - Open-Meteo's archive
    # and elevation endpoints are unauthenticated - so a missing .env is a
    # warning, not a failure.
    print("No .env found; none of the sources in this notebook require a key.")

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "pm25"
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
BASE_DIR = PROCESSED_DIR / "base"
SAT_CACHE = RAW_DIR / "satellite"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "pm25"
PHASE7_DIR = ARTIFACT_DIR / "phase7"
for p in [RAW_DIR, PROCESSED_DIR, BASE_DIR, SAT_CACHE, PHASE7_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # CAMS variables. aerosol_optical_depth is the headline one: it is the
    # column-integrated aerosol signal that published hyper-local PM2.5 models
    # rely on for coverage BETWEEN stations.
    "cams_vars": [
        "pm2_5", "pm10", "aerosol_optical_depth", "nitrogen_dioxide",
        "carbon_monoxide", "sulphur_dioxide", "dust", "ozone",
    ],
    # Terrain roughness is sampled on a ring around each station. 10 km is the
    # scale at which valley-vs-plain changes dispersion.
    "terrain_ring_km": 10.0,
    "terrain_ring_points": 8,
    "throttle_s": float(os.getenv("DRIVER_THROTTLE_S", "0.15")),
    "schema_version": "1.0.0",
}

print("Project  :", PROJECT_ROOT)
print("Data root:", DATA_ROOT)

Loaded env from: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/.env
Project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
Data root: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25


/Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Station coordinates and window, taken from Phase 1

The driver layers must be keyed to exactly the coordinates and time span the target
layer uses. Reading both from `base_dataset.parquet` rather than re-deriving them
means a Phase-1 rescope propagates automatically instead of producing a driver layer
that misses half the stations.

In [2]:
# ============================================================================
# STATION COORDINATES AND WINDOW - derived from the Phase-1 target layer
# ============================================================================
BASE_FILE = BASE_DIR / "base_dataset.parquet"
if not BASE_FILE.exists():
    raise FileNotFoundError(f"{BASE_FILE} not found. Run Phase 1 first.")

# Only the columns needed to define scope - the full frame is 1.7M rows and is
# not required here.
scope = pd.read_parquet(BASE_FILE, columns=["timestamp_utc", "location_id",
                                            "station_name", "lat", "lon"])
scope["timestamp_utc"] = pd.to_datetime(scope["timestamp_utc"], utc=True)

WINDOW_START = scope.timestamp_utc.min().tz_convert("UTC").normalize()
WINDOW_END = scope.timestamp_utc.max().tz_convert("UTC").normalize()

stations = (scope.groupby(["lat", "lon"], as_index=False)
                 .agg(location_id=("location_id", "first"),
                      station_name=("station_name", "first"),
                      rows=("timestamp_utc", "size")))
# Round to the same 4dp the Phase-1 weather cache uses, so a coordinate maps to
# one cache file in both notebooks.
stations["lat"] = stations["lat"].round(4)
stations["lon"] = stations["lon"].round(4)
stations = stations.drop_duplicates(subset=["lat", "lon"]).reset_index(drop=True)

print(f"Target layer : {len(scope):,} rows, {scope.location_id.nunique()} stations")
print(f"Window       : {WINDOW_START.date()} -> {WINDOW_END.date()} "
      f"({(WINDOW_END - WINDOW_START).days + 1} days)")
print(f"Coordinates  : {len(stations)} unique to acquire")
del scope
display(stations.head())

Target layer : 1,705,252 rows, 149 stations
Window       : 2025-02-18 -> 2026-09-09 (569 days)
Coordinates  : 149 unique to acquire


,lat,lon,location_id,station_name,rows
0,13.0878,80.2785,8558,Chennai,6079
1,13.6154,79.4092,3409373,"Vaikuntapuram, Tirupathi - APPCB",10656
2,14.6759,77.5930,358456,"Gulzarpet, Anantapur - APPCB",11404
3,15.3715,75.1160,3409387,"Lingaraj Nagar, Hubballi - KSPCB",11702
4,16.5097,80.6122,3409493,"Rajiv Gandhi Park, Vijayawada - APPCB",9162


## 2. CAMS atmospheric composition adapter

One call per coordinate covers the whole window — the archive returns ~13.6k hourly
values in about a second — so the acquisition is ~150 requests, not 150 × months.

The adapter shape mirrors the Phase-1 weather adapter (`fetch` → `normalize` →
canonical columns) so a different composition provider (Sentinel-5P via Copernicus,
or MERRA-2) can be substituted without touching Phase 2.

In [3]:
# ============================================================================
# CAMS COMPOSITION ADAPTER - Open-Meteo air-quality archive
# ============================================================================
# PROVENANCE WARNING, repeated here because this is the cell someone copies:
# these values are CAMS reanalysis OUTPUT. CAMS is a chemical transport model.
# Its pm2_5 field is a model estimate of the same quantity our target layer
# MEASURES. Treating it as an observation - or scoring against it and calling
# the result station accuracy - would be wrong.
#
# The 'domains=cams_global' pin matters: without it Open-Meteo may serve a
# regional domain for some coordinates and the global one for others, which
# would put two different models in one column.
# ============================================================================
OPEN_METEO_AQ = "https://air-quality-api.open-meteo.com/v1/air-quality"

# Canonical names. The cams_ prefix is not cosmetic - it is what lets Phase 2
# select or drop the entire model-derived block in one operation, which the
# A0-A6 ablation needs.
CAMS_RENAME = {
    "pm2_5": "cams_pm25",
    "pm10": "cams_pm10",
    "aerosol_optical_depth": "cams_aod",
    "nitrogen_dioxide": "cams_no2",
    "carbon_monoxide": "cams_co",
    "sulphur_dioxide": "cams_so2",
    "dust": "cams_dust",
    "ozone": "cams_o3",
}
CAMS_COLUMNS = ["timestamp_utc", "lat", "lon"] + list(CAMS_RENAME.values()) + ["satellite_source"]


def _http_json(url, params, retries=5, backoff=2.0, timeout=(10, 60)):
    """GET with retry on the transient statuses Open-Meteo actually returns.

    Phase 1 lost 144 of 149 stations to a transient DNS failure that was not
    retried, so the retry loop is deliberately patient.

    The timeout is a (connect, read) PAIR rather than a single large number.
    A single 180s value made one hung connection cost 6 x 180s before the loop
    could move on, which stalled the whole acquisition on one coordinate.
    """
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(str(r.status_code), response=r)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError) as e:
            last = e
            if attempt == retries - 1:
                break
            time.sleep(backoff * (2 ** attempt))
    raise last


def fetch_cams(lat, lon, start, end):
    payload = _http_json(OPEN_METEO_AQ, {
        "latitude": lat, "longitude": lon,
        "start_date": pd.Timestamp(start).date().isoformat(),
        "end_date": pd.Timestamp(end).date().isoformat(),
        "hourly": ",".join(CONFIG["cams_vars"]),
        "timezone": "UTC",
        "domains": "cams_global",
    })
    df = pd.DataFrame(payload.get("hourly", {}))
    if "time" in df:
        df["timestamp_utc"] = pd.to_datetime(df.pop("time"), utc=True)
    # The response carries the DEM elevation of the grid cell - free, and
    # reused by the terrain section below.
    return df, payload.get("elevation")


def normalize_cams(df, lat, lon, source_name="open-meteo-cams-global"):
    """Map the provider response into AeroPulse's composition contract."""
    out = df.rename(columns=CAMS_RENAME).copy()
    out["lat"], out["lon"] = lat, lon
    out["satellite_source"] = source_name
    for c in CAMS_COLUMNS:
        if c not in out.columns:
            out[c] = np.nan
    return out[CAMS_COLUMNS]


print("CAMS adapter ready. Variables:", list(CAMS_RENAME.values()))

CAMS adapter ready. Variables: ['cams_pm25', 'cams_pm10', 'cams_aod', 'cams_no2', 'cams_co', 'cams_so2', 'cams_dust', 'cams_o3']


### Acquisition

Cached per coordinate, so a network failure costs one station rather than the run, and
a re-execution of this notebook is free.

In [4]:
# ============================================================================
# CAMS ACQUISITION - one call per coordinate, cached
# ============================================================================
t0 = time.time()
parts, elevations, failed = [], {}, []

for rec in tqdm(stations.to_dict(orient="records"), desc="CAMS"):
    cache = SAT_CACHE / f"cams_{rec['lat']:.4f}_{rec['lon']:.4f}.parquet"
    if cache.exists():
        parts.append(pd.read_parquet(cache))
        continue
    try:
        raw, elev = fetch_cams(rec["lat"], rec["lon"], WINDOW_START, WINDOW_END)
        if raw.empty:
            failed.append((rec["lat"], rec["lon"], "empty response"))
            continue
        norm = normalize_cams(raw, rec["lat"], rec["lon"])
        norm.to_parquet(cache, index=False)
        parts.append(norm)
        if elev is not None:
            elevations[(rec["lat"], rec["lon"])] = float(elev)
        time.sleep(CONFIG["throttle_s"])
    except Exception as e:
        failed.append((rec["lat"], rec["lon"], type(e).__name__))
        print(f"  CAMS failed at {rec['lat']:.3f},{rec['lon']:.3f}: {type(e).__name__}")

satellite = (pd.concat(parts, ignore_index=True) if parts
             else pd.DataFrame(columns=CAMS_COLUMNS))

print(f"\nCAMS rows: {len(satellite):,} for {len(stations) - len(failed)}/{len(stations)} "
      f"coordinates in {time.time() - t0:.0f}s")
if failed:
    print(f"FAILED for {len(failed)} coordinates - rerun this cell to retry only "
          f"those (successes are cached):")
    for f in failed[:10]:
        print("   ", f)

CAMS:   0%|          | 0/149 [00:00<?, ?it/s]

CAMS:  47%|████▋     | 70/149 [00:00<00:00, 695.09it/s]

CAMS: 100%|██████████| 149/149 [00:00<00:00, 750.71it/s]

CAMS: 100%|██████████| 149/149 [00:00<00:00, 740.70it/s]


CAMS rows: 2,034,744 for 149/149 coordinates in 0s


## 3. Normalize and persist the composition layer

Two checks before writing, both of which have caught real acquisition faults in this
project: coverage (did every station get data?) and physical plausibility (is the
column actually what it claims to be?).

In [5]:
# ============================================================================
# PERSIST the composition layer
# ============================================================================
VALUE_COLS = list(CAMS_RENAME.values())

for c in VALUE_COLS:
    satellite[c] = pd.to_numeric(satellite[c], errors="coerce").astype("float32")
satellite["timestamp_utc"] = pd.to_datetime(satellite["timestamp_utc"], utc=True)
satellite = satellite.drop_duplicates(subset=["timestamp_utc", "lat", "lon"])
satellite = satellite.sort_values(["lat", "lon", "timestamp_utc"]).reset_index(drop=True)

print(f"Rows      : {len(satellite):,}")
print(f"Coords    : {satellite.groupby(['lat','lon']).ngroups}")
print(f"Span      : {satellite.timestamp_utc.min()} -> {satellite.timestamp_utc.max()}")
print("\nNull rate and range per variable:")
for c in VALUE_COLS:
    s = satellite[c]
    print(f"  {c:12} null {s.isna().mean():6.2%}   "
          f"min {np.nanmin(s):9.2f}  median {np.nanmedian(s):9.2f}  max {np.nanmax(s):10.2f}")

# Plausibility gates. AOD is dimensionless and effectively bounded near 0-5 over
# land; a value outside that means the column has been mis-mapped, which is the
# failure mode a null-rate check alone would miss.
aod = satellite["cams_aod"].dropna()
if len(aod) and (aod.min() < -0.01 or aod.max() > 10):
    raise ValueError(f"cams_aod out of physical range [{aod.min()}, {aod.max()}] "
                     "- check the variable mapping.")
neg = {c: int((satellite[c] < 0).sum()) for c in VALUE_COLS}
if any(neg.values()):
    print("\nWARNING - negative concentrations (should be zero):", 
          {k: v for k, v in neg.items() if v})

SAT_FILE = BASE_DIR / "satellite_hourly.parquet"
satellite.to_parquet(SAT_FILE, index=False)
print(f"\nWrote: {SAT_FILE}  ({SAT_FILE.stat().st_size/1e6:.1f} MB)")

Rows      : 2,034,744
Coords    : 149
Span      : 2025-02-18 00:00:00+00:00 -> 2026-09-09 23:00:00+00:00

Null rate and range per variable:
  cams_pm25    null  0.00%   min      0.20  median     29.00  max     814.00
  cams_pm10    null  0.00%   min      0.20  median     42.40  max    3270.80
  cams_aod     null  0.00%   min      0.03  median      0.45  max       4.41
  cams_no2     null  0.00%   min      0.00  median      9.20  max     220.10
  cams_co      null  0.00%   min     58.00  median    298.00  max    4984.00
  cams_so2     null  0.00%   min      0.00  median      9.50  max     162.90
  cams_dust    null  0.00%   min      0.00  median     11.00  max    9119.00
  cams_o3      null  0.00%   min      0.00  median     78.00  max     523.00



Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/base/satellite_hourly.parquet  (23.1 MB)


### Sanity check — does the model layer track the observed one?

CAMS PM2.5 and station PM2.5 are estimates of the same quantity, so their correlation is
a genuine check that the join keys and time alignment are right: an uncorrelated pair
would mean the coordinates or timestamps are mismatched.

This number is **not** a model score and must never be quoted as one. It is measured
here precisely so that `cams_pm25` can be excluded from the feature set with evidence
rather than by assertion.

In [6]:
# ============================================================================
# CAMS-vs-observation diagnostic  (alignment check, NOT a model metric)
# ============================================================================
# Why this is not an accuracy claim: CAMS assimilates satellite data and runs a
# transport model; comparing it to stations measures how well ECMWF's model does
# on this network, and says nothing about AeroPulse. It is used here for exactly
# one purpose - confirming the join is aligned.
# ============================================================================
probe_cols = ["timestamp_utc", "lat", "lon", "pm25"]
probe = pd.read_parquet(BASE_FILE, columns=probe_cols)
probe["timestamp_utc"] = pd.to_datetime(probe["timestamp_utc"], utc=True)
probe["lat"] = probe["lat"].round(4)
probe["lon"] = probe["lon"].round(4)

# Observations land on HH:30 for IST stations while CAMS is on HH:00, so an
# equality join drops everything - the same trap Phase 1 documented for weather.
# merge_asof backward takes the most recent composition hour AT OR BEFORE each
# observation, which is also the only direction that cannot leak.
merged = pd.merge_asof(
    probe.sort_values("timestamp_utc"),
    satellite[["timestamp_utc", "lat", "lon", "cams_pm25", "cams_aod"]].sort_values("timestamp_utc"),
    on="timestamp_utc", by=["lat", "lon"],
    direction="backward", tolerance=pd.Timedelta("90min"),
)
cov = merged["cams_pm25"].notna().mean()
pair = merged.dropna(subset=["pm25", "cams_pm25"])
print(f"Join coverage           : {cov:.1%}")
print(f"Paired rows             : {len(pair):,}")
if len(pair) > 1000:
    r_pm = pair["pm25"].corr(pair["cams_pm25"])
    r_aod = pair["pm25"].corr(pair["cams_aod"])
    print(f"corr(observed, cams_pm25): {r_pm:.3f}   <- alignment check only")
    print(f"corr(observed, cams_aod) : {r_aod:.3f}   <- the feature we actually want")
    print(f"CAMS bias vs observed    : {(pair['cams_pm25'] - pair['pm25']).mean():+.1f} ug/m3")
    if r_pm < 0.2:
        print("\nWARNING: near-zero correlation suggests a coordinate or timestamp "
              "misalignment, not a bad model. Check the join keys.")
if cov < 0.90:
    print(f"\nWARNING: only {cov:.1%} of observations found a composition hour.")
del probe, merged, pair

Join coverage           : 100.0%
Paired rows             : 1,705,252
corr(observed, cams_pm25): 0.361   <- alignment check only
corr(observed, cams_aod) : 0.043   <- the feature we actually want
CAMS bias vs observed    : +1.6 ug/m3


## 4. Static terrain

Elevation explains part of why two nearby stations differ persistently, and terrain
**roughness** explains more: a station in a bowl ventilates differently from one on a
plain even at identical elevation. Roughness is measured by sampling a ring of points
around each station rather than being assumed from a land-use class.

`is_indo_gangetic_plain` is a coarse bounding box, and is labelled as such — it is a
crude proxy for the airshed that produces India's severe winter episodes, not a
gazetteer lookup.

In [7]:
# ============================================================================
# STATIC TERRAIN - elevation and 10 km roughness
# ============================================================================
# The elevation endpoint takes up to 100 coordinates per call, so the ring
# samples are batched rather than fetched one at a time.
# ============================================================================
OPEN_METEO_ELEV = "https://api.open-meteo.com/v1/elevation"
EARTH_R_KM = 6371.0088


def ring_coords(lat, lon, radius_km, n_points):
    """n evenly-spaced points on a circle of radius_km around (lat, lon)."""
    out = []
    for b in np.linspace(0, 2 * np.pi, n_points, endpoint=False):
        dlat = np.rad2deg(radius_km / EARTH_R_KM * np.cos(b))
        dlon = np.rad2deg(radius_km / EARTH_R_KM * np.sin(b) /
                          max(np.cos(np.deg2rad(lat)), 1e-6))
        out.append((lat + dlat, lon + dlon))
    return out


def fetch_elevations(coords, chunk=100):
    """Batch the elevation lookup; returns a list aligned with `coords`."""
    values = []
    for i in range(0, len(coords), chunk):
        block = coords[i:i + chunk]
        payload = _http_json(OPEN_METEO_ELEV, {
            "latitude": ",".join(f"{c[0]:.5f}" for c in block),
            "longitude": ",".join(f"{c[1]:.5f}" for c in block),
        }, timeout=(10, 60))
        values.extend(payload.get("elevation", [np.nan] * len(block)))
        time.sleep(CONFIG["throttle_s"])
    return values


ELEV_CACHE = SAT_CACHE / "station_terrain.parquet"
if ELEV_CACHE.exists():
    terrain = pd.read_parquet(ELEV_CACHE)
    print(f"Terrain loaded from cache: {len(terrain)} stations")
else:
    t0 = time.time()
    centres = [(float(r.lat), float(r.lon)) for r in stations.itertuples()]
    ring = []
    for lat, lon in centres:
        ring.extend(ring_coords(lat, lon, CONFIG["terrain_ring_km"],
                                CONFIG["terrain_ring_points"]))

    centre_elev = fetch_elevations(centres)
    ring_elev = np.asarray(fetch_elevations(ring), dtype="float64").reshape(
        len(centres), CONFIG["terrain_ring_points"])

    terrain = stations[["location_id", "station_name", "lat", "lon"]].copy()
    terrain["elevation_m"] = np.asarray(centre_elev, dtype="float64")
    terrain["elevation_std_10km"] = np.nanstd(ring_elev, axis=1)
    terrain["elevation_range_10km"] = (np.nanmax(ring_elev, axis=1) -
                                       np.nanmin(ring_elev, axis=1))
    # Positive => the station sits in a hollow relative to its surroundings,
    # which impedes ventilation.
    terrain["elevation_relief_10km"] = (np.nanmean(ring_elev, axis=1) -
                                        terrain["elevation_m"])
    terrain.to_parquet(ELEV_CACHE, index=False)
    print(f"Terrain fetched for {len(terrain)} stations in {time.time()-t0:.0f}s")

# Coarse Indo-Gangetic Plain box. Documented as an approximation: the IGP is the
# airshed behind India's severe winter and post-monsoon episodes, and a station
# inside it behaves differently from one on the Deccan at the same elevation.
terrain["is_indo_gangetic_plain"] = (
    terrain.lat.between(22.0, 32.0) & terrain.lon.between(73.0, 89.0)
).astype("int8")

GEO_FILE = BASE_DIR / "static_geography.parquet"
terrain.to_parquet(GEO_FILE, index=False)

print(f"\nelevation_m         : {terrain.elevation_m.min():.0f} .. "
      f"{terrain.elevation_m.max():.0f} m (median {terrain.elevation_m.median():.0f})")
print(f"elevation_std_10km  : median {terrain.elevation_std_10km.median():.1f} m, "
      f"max {terrain.elevation_std_10km.max():.1f} m")
print(f"Indo-Gangetic Plain : {terrain.is_indo_gangetic_plain.sum()}/{len(terrain)} stations")
print(f"\nWrote: {GEO_FILE}")
display(terrain.sort_values("elevation_m", ascending=False).head())

Terrain loaded from cache: 149 stations

elevation_m         : 2 .. 798 m (median 218)
elevation_std_10km  : median 22.0 m, max 256.6 m
Indo-Gangetic Plain : 67/149 stations

Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/base/static_geography.parquet


,location_id,station_name,lat,lon,elevation_m,elevation_std_10km,elevation_range_10km,elevation_relief_10km,is_indo_gangetic_plain
72,3409384,"Barsua Iron Ore Mines, Tensa - OSPCB",21.8700,85.1670,798.0,256.561853,749.0,-190.625,0
50,3409453,"MIDC Ambad, Nashik - MPCB",19.9502,73.7315,666.0,49.951852,165.0,-41.250,0
19,3410005,"Dhankawadi, Pune - IITM",18.4599,73.8522,659.0,117.462161,343.0,27.875,0
18,3409452,"Sawe Wadi, Latur - MPCB",18.3996,76.5745,628.0,14.378804,53.0,-22.000,0
3,3409387,"Lingaraj Nagar, Hubballi - KSPCB",15.3715,75.1160,625.0,27.986325,89.0,14.625,0


## 5. Festival and crop-burning calendar

`sin_doy` / `cos_doy` cannot represent Diwali. Diwali moves through the Gregorian
calendar — 12 Nov 2023, 31 Oct 2024, 20 Oct 2025, 8 Nov 2026 — so an annual sinusoid
places it in a different phase each year and the model cannot learn it. Yet it produces
one of the sharpest PM2.5 spikes in northern India.

The same argument applies to crop-residue burning: the kharif (paddy) window in October
and November is the driver of the post-monsoon episodes this product exists to warn
about, and the rabi (wheat) window in April and May is a weaker second season.

This table is **curated and hand-authored**, not fetched. That is a deliberate
trade-off: it is auditable and has no runtime dependency, but it must be extended by
hand beyond its coverage. The gate at the end of this section fails loudly if the
dataset window runs past the table.

In [8]:
# ============================================================================
# FESTIVAL / CROP-BURNING CALENDAR - curated, deterministic, no API
# ============================================================================
# Sources are the published Gregorian dates for each observance. Movable feasts
# are listed explicitly per year because that is the whole point: a formula
# cannot place them, which is why sin_doy/cos_doy fail on exactly these days.
# ============================================================================
DIWALI = {                      # Lakshmi Puja / main Diwali night
    2023: "2023-11-12", 2024: "2024-10-31", 2025: "2025-10-20",
    2026: "2026-11-08", 2027: "2027-10-29",
}
HOLI = {
    2023: "2023-03-08", 2024: "2024-03-25", 2025: "2025-03-14",
    2026: "2026-03-04", 2027: "2027-03-22",
}
DUSSEHRA = {
    2023: "2023-10-24", 2024: "2024-10-12", 2025: "2025-10-02",
    2026: "2026-10-20", 2027: "2027-10-09",
}
GURU_NANAK = {
    2023: "2023-11-27", 2024: "2024-11-15", 2025: "2025-11-05",
    2026: "2026-11-24", 2027: "2027-11-14",
}
EID_FITR = {
    2023: "2023-04-22", 2024: "2024-04-11", 2025: "2025-03-31",
    2026: "2026-03-20", 2027: "2027-03-10",
}
EID_ADHA = {
    2023: "2023-06-29", 2024: "2024-06-17", 2025: "2025-06-07",
    2026: "2026-05-27", 2027: "2027-05-17",
}
FIXED_HOLIDAYS = {              # (month, day) -> name
    (1, 1): "new_year", (1, 26): "republic_day", (8, 15): "independence_day",
    (10, 2): "gandhi_jayanti", (12, 25): "christmas",
}
# Crop-residue burning windows (month-day ranges, inclusive).
#   kharif = paddy stubble, Punjab/Haryana. The severe one.
#   rabi   = wheat stubble. Weaker, and competes with pre-monsoon dust.
BURNING_WINDOWS = {"kharif": ((10, 1), (11, 30)), "rabi": ((4, 1), (5, 31))}

CALENDAR_YEARS = sorted(DIWALI)
_covered = pd.Timestamp(f"{CALENDAR_YEARS[-1]}-12-31").date()
_needed = WINDOW_END.date()
if _needed > _covered:
    raise ValueError(
        f"Calendar table covers to {_covered} but the dataset runs to {_needed}. "
        "Extend DIWALI/HOLI/... above - do not let this silently emit zeros.")


def _in_window(ts, window):
    (m0, d0), (m1, d1) = window
    lo = pd.Timestamp(year=ts.year, month=m0, day=d0)
    hi = pd.Timestamp(year=ts.year, month=m1, day=d1)
    return lo <= ts <= hi


# One row per LOCAL date across the dataset span plus a margin, so a lag or a
# forward-looking target never falls off the end of the table.
dates = pd.date_range(WINDOW_START.tz_convert("Asia/Kolkata").normalize().tz_localize(None)
                      - pd.Timedelta(days=10),
                      WINDOW_END.tz_convert("Asia/Kolkata").normalize().tz_localize(None)
                      + pd.Timedelta(days=10), freq="D")
cal = pd.DataFrame({"date": dates})

diwali_dates = pd.to_datetime(sorted(DIWALI.values()))
festival_map = {"holi": HOLI, "dussehra": DUSSEHRA, "guru_nanak": GURU_NANAK,
                "eid_fitr": EID_FITR, "eid_adha": EID_ADHA, "diwali": DIWALI}

# days_to / days_after are separate non-negative columns rather than one signed
# column: the PM2.5 response is asymmetric - firework smoke persists for days
# AFTER the night - and a single signed feature forces a tree to split twice to
# express that.
_d = cal["date"].to_numpy()
_f = diwali_dates.to_numpy()
nxt = np.searchsorted(_f, _d, side="left")
prv = np.clip(nxt - 1, 0, len(_f) - 1)
nxt_c = np.clip(nxt, 0, len(_f) - 1)
_day = lambda td: td.astype("timedelta64[D]").astype("int32")
cal["days_to_diwali"] = np.clip(_day(_f[nxt_c] - _d), 0, None)
cal["days_after_diwali"] = np.clip(_day(_d - _f[prv]), 0, None)
cal["diwali_window"] = ((cal.days_to_diwali <= 3) | (cal.days_after_diwali <= 7)).astype("int8")

all_festivals = pd.to_datetime(sorted({d for m in festival_map.values() for d in m.values()}))
fest_delta = np.abs((cal["date"].to_numpy()[:, None] -
                     all_festivals.to_numpy()[None, :]).astype("timedelta64[D]").astype(int))
cal["festival_window"] = (fest_delta.min(axis=1) <= 3).astype("int8")
cal["days_to_nearest_festival"] = fest_delta.min(axis=1).astype("int16")

holiday_name = []
for ts in cal["date"]:
    name = FIXED_HOLIDAYS.get((ts.month, ts.day))
    if name is None:
        for fname, fmap in festival_map.items():
            if fmap.get(ts.year) and pd.Timestamp(fmap[ts.year]) == ts:
                name = fname
                break
    holiday_name.append(name or "")
cal["holiday_name"] = holiday_name
cal["is_holiday"] = (cal.holiday_name != "").astype("int8")

for label, window in BURNING_WINDOWS.items():
    cal[f"crop_burning_{label}"] = cal["date"].map(
        lambda ts, w=window: _in_window(ts, w)).astype("int8")
cal["crop_burning_window"] = (cal.crop_burning_kharif | cal.crop_burning_rabi).astype("int8")

CAL_FILE = BASE_DIR / "calendar_events.parquet"
cal.to_parquet(CAL_FILE, index=False)

print(f"Calendar rows : {len(cal):,}  ({cal.date.min().date()} -> {cal.date.max().date()})")
print(f"  holidays            : {cal.is_holiday.sum()}")
print(f"  diwali_window days  : {cal.diwali_window.sum()}")
print(f"  festival_window days: {cal.festival_window.sum()}")
print(f"  kharif burning days : {cal.crop_burning_kharif.sum()}")
print(f"  rabi burning days   : {cal.crop_burning_rabi.sum()}")
print(f"\nWrote: {CAL_FILE}")
display(cal[cal.diwali_window == 1].head(12))

Calendar rows : 589  (2025-02-08 -> 2026-09-19)
  holidays            : 14
  diwali_window days  : 11
  festival_window days: 63
  kharif burning days : 61
  rabi burning days   : 122

Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25/processed/base/calendar_events.parquet


,date,days_to_diwali,days_after_diwali,diwali_window,festival_window,days_to_nearest_festival,holiday_name,is_holiday,crop_burning_kharif,crop_burning_rabi,crop_burning_window
251,2025-10-17,3,351,1,1,3,,0,1,0,1
252,2025-10-18,2,352,1,1,2,,0,1,0,1
253,2025-10-19,1,353,1,1,1,,0,1,0,1
254,2025-10-20,0,354,1,1,0,diwali,1,1,0,1
255,2025-10-21,383,1,1,1,1,,0,1,0,1
256,2025-10-22,382,2,1,1,2,,0,1,0,1
257,2025-10-23,381,3,1,1,3,,0,1,0,1
258,2025-10-24,380,4,1,0,4,,0,1,0,1
259,2025-10-25,379,5,1,0,5,,0,1,0,1
260,2025-10-26,378,6,1,0,6,,0,1,0,1


### Does the calendar actually mark a PM2.5 signal?

A calendar feature is only worth its complexity if the days it flags behave differently.
This check is what distinguishes a useful feature from a decorative one — and it runs
against observed PM2.5, so it is a statement about the data, not about a model.

In [9]:
# ============================================================================
# CALENDAR VALIDATION - do the flagged days differ from ordinary ones?
# ============================================================================
check = pd.read_parquet(BASE_FILE, columns=["timestamp_local", "pm25", "lat", "lon"])
check["date"] = pd.to_datetime(check["timestamp_local"]).dt.tz_localize(None).dt.normalize()
check = check.merge(cal, on="date", how="left")

baseline = check.loc[check.diwali_window.eq(0) & check.crop_burning_window.eq(0), "pm25"].mean()
print(f"Ordinary days           : mean PM2.5 {baseline:6.1f} ug/m3")
for flag in ["diwali_window", "festival_window", "crop_burning_kharif",
             "crop_burning_rabi", "is_holiday"]:
    sel = check.loc[check[flag].eq(1), "pm25"]
    if len(sel):
        print(f"{flag:24}: mean PM2.5 {sel.mean():6.1f} ug/m3  "
              f"({sel.mean()/baseline - 1:+6.1%} vs ordinary, n={len(sel):,})")

# Day-by-day around Diwali. The spike should be ON and just AFTER the night,
# which is what justifies keeping days_after_diwali as its own column.
near = check[check.days_to_diwali.le(5) | check.days_after_diwali.le(5)].copy()
near["offset"] = np.where(near.days_to_diwali.le(5), -near.days_to_diwali, near.days_after_diwali)
prof = near.groupby("offset")["pm25"].agg(["mean", "size"]).round(1)
print("\nPM2.5 by day relative to Diwali (0 = Diwali night):")
print(prof.to_string())
del check, near

Ordinary days           : mean PM2.5   34.9 ug/m3
diwali_window           : mean PM2.5   56.6 ug/m3  (+62.2% vs ordinary, n=33,663)
festival_window         : mean PM2.5   40.5 ug/m3  (+16.1% vs ordinary, n=197,646)
crop_burning_kharif     : mean PM2.5   51.8 ug/m3  (+48.5% vs ordinary, n=189,773)
crop_burning_rabi       : mean PM2.5   34.0 ug/m3  ( -2.6% vs ordinary, n=363,084)
is_holiday              : mean PM2.5   43.0 ug/m3  (+23.2% vs ordinary, n=43,625)

PM2.5 by day relative to Diwali (0 = Diwali night):
        mean  size
offset            
-5      49.7  3010
-4      48.8  3319
-3      50.7  3343
-2      57.3  3360
-1      61.9  3286
 0      81.4  2506
 1      84.9  3372
 2      59.3  3385
 3      52.5  3397
 4      46.7  3311
 5      42.6  3353


## 6. Quality gates and manifest

Fail here rather than three notebooks downstream. Each gate corresponds to a failure
this pipeline has actually produced or is one Phase 2 cannot detect on its own.

In [10]:
# ============================================================================
# DRIVER-LAYER QUALITY GATES
# ============================================================================
gates, failures = {}, []


def gate(name, ok, detail=""):
    gates[name] = {"pass": bool(ok), "detail": detail}
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:38} {detail}")
    if not ok:
        failures.append(name)


print("Driver-layer gates:")
sat_coords = satellite.groupby(["lat", "lon"]).ngroups
gate("satellite covers all stations", sat_coords == len(stations),
     f"{sat_coords}/{len(stations)} coordinates")
gate("satellite spans target window",
     satellite.timestamp_utc.min() <= WINDOW_START and
     satellite.timestamp_utc.max() >= WINDOW_END,
     f"{satellite.timestamp_utc.min().date()} -> {satellite.timestamp_utc.max().date()}")
gate("no duplicate coord-hours",
     not satellite.duplicated(subset=["timestamp_utc", "lat", "lon"]).any())
gate("cams_aod populated", satellite.cams_aod.notna().mean() > 0.95,
     f"{satellite.cams_aod.notna().mean():.1%} non-null")
gate("terrain covers all stations", len(terrain) == len(stations),
     f"{len(terrain)}/{len(stations)}")
gate("elevation plausible for India",
     terrain.elevation_m.between(-100, 5000).all(),
     f"{terrain.elevation_m.min():.0f}..{terrain.elevation_m.max():.0f} m")
gate("calendar spans target window",
     cal.date.min() <= WINDOW_START.tz_localize(None) and
     cal.date.max() >= WINDOW_END.tz_localize(None))
gate("diwali marked in window", cal.diwali_window.sum() > 0,
     f"{cal.diwali_window.sum()} days")

if failures:
    raise RuntimeError(f"Driver acquisition gates failed: {failures}")
print("\nAll gates passed.")

Driver-layer gates:


  [PASS] satellite covers all stations          149/149 coordinates
  [PASS] satellite spans target window          2025-02-18 -> 2026-09-09


  [PASS] no duplicate coord-hours               
  [PASS] cams_aod populated                     100.0% non-null
  [PASS] terrain covers all stations            149/149
  [PASS] elevation plausible for India          2..798 m
  [PASS] calendar spans target window           
  [PASS] diwali marked in window                11 days

All gates passed.


In [11]:
# ============================================================================
# MANIFEST - provenance, and an explicit record of what is still missing
# ============================================================================
def _fingerprint(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()[:16]


manifest = {
    "schema_version": CONFIG["schema_version"],
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "window": [str(WINDOW_START), str(WINDOW_END)],
    "stations": int(len(stations)),
    "layers": {
        "satellite": {
            "file": str(SAT_FILE.relative_to(PROJECT_ROOT)),
            "rows": int(len(satellite)),
            "columns": VALUE_COLS,
            "source": "open-meteo air-quality archive, domains=cams_global",
            "provenance": "CAMS reanalysis MODEL OUTPUT - not ground truth",
            "usage_policy": [
                "predictor only; never a target",
                "no metric against these columns may be called station accuracy",
                "cams_pm25 is audit-only and excluded from the feature contract",
            ],
            "sha256_16": _fingerprint(SAT_FILE),
        },
        "static_geography": {
            "file": str(GEO_FILE.relative_to(PROJECT_ROOT)),
            "rows": int(len(terrain)),
            "columns": ["elevation_m", "elevation_std_10km", "elevation_range_10km",
                        "elevation_relief_10km", "is_indo_gangetic_plain"],
            "source": "open-meteo elevation (Copernicus DEM); IGP box is an approximation",
            "sha256_16": _fingerprint(GEO_FILE),
        },
        "calendar": {
            "file": str(CAL_FILE.relative_to(PROJECT_ROOT)),
            "rows": int(len(cal)),
            "columns": ["is_holiday", "diwali_window", "days_to_diwali",
                        "days_after_diwali", "festival_window",
                        "days_to_nearest_festival", "crop_burning_kharif",
                        "crop_burning_rabi", "crop_burning_window"],
            "source": "curated in-notebook table",
            "covers_years": CALENDAR_YEARS,
            "sha256_16": _fingerprint(CAL_FILE),
        },
    },
    "gates": gates,
    "open_gaps": [
        "population_density - needs WorldPop/GHSL raster",
        "road_density, distance_to_major_road - needs OSM extract",
        "land_use, urban_fraction, building_density - needs GHSL/ESA WorldCover",
        "distance_to_industrial_area - no open authoritative source identified",
    ],
    "excluded_from_features": {
        "cams_pm25": "model estimate of the target; would create a shortcut that "
                     "scores well and does not transfer",
        "cams_pm10": "retained as a feature - it is a different quantity from the "
                     "target - but flagged as model output",
    },
}

MANIFEST_FILE = PHASE7_DIR / "driver_acquisition_manifest.json"
with open(MANIFEST_FILE, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print("DRIVER ACQUISITION COMPLETE\n")
for name, layer in manifest["layers"].items():
    print(f"  {name:18} {layer['rows']:>10,} rows   sha {layer['sha256_16']}")
print(f"\nWrote: {MANIFEST_FILE}")
print("\nPhase 2 consumes satellite_hourly / static_geography / calendar_events.")
print("Still missing (recorded in open_gaps):")
for g in manifest["open_gaps"]:
    print("   -", g)

DRIVER ACQUISITION COMPLETE

  satellite           2,034,744 rows   sha afbfe7749b2c148f
  static_geography          149 rows   sha c23ca42764539ef7
  calendar                  589 rows   sha f2454fb2aec59afa

Wrote: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/artifacts/pm25/phase7/driver_acquisition_manifest.json

Phase 2 consumes satellite_hourly / static_geography / calendar_events.
Still missing (recorded in open_gaps):
   - population_density - needs WorldPop/GHSL raster
   - road_density, distance_to_major_road - needs OSM extract
   - land_use, urban_fraction, building_density - needs GHSL/ESA WorldCover
   - distance_to_industrial_area - no open authoritative source identified
